# 🔌 ISOM 260 · Your Tool, Plugged In

**MCP (Model Context Protocol)** in five minutes, using the weather tool you already know.

In Session 4 you gave your agent a tool by hand: a Python function **plus** a job description you wrote yourself, wired into *your* loop. It worked for your agent and nobody else's.

Today that tool becomes an **MCP server**, and *any* agent can plug in: Gemini, Claude, ChatGPT, or your phone bot. You already acted out the whole protocol in class:

| | the agent says | the server says |
|---|---|---|
| ① | *"What can you do?"* | "Here's my menu." |
| ② | *"Do this."* | (does it) |
| ③ | | "Here's the result." |

Today you'll see those exact messages cross the wire. **Run the cells top to bottom.** You need `GOOGLE_API_KEY` in Colab Secrets (🔑 on the left).

In [ ]:
!pip install -q fastmcp google-genai

In [ ]:
from google.colab import userdata
from IPython.display import display, HTML
import html as _h

class Stop(Exception):
    """A friendly stop: one clear message instead of a wall of red."""
    def _render_traceback_(self): return ["⛔ " + str(self)]

def help_box(title, steps):
    display(HTML("<div style='border-left:5px solid #b42318;background:#fff5f5;color:#1d2226;padding:12px 16px;border-radius:6px;"
                 "font-family:Arial,sans-serif;font-size:14px;max-width:640px'><b>😵 " + title + "</b><ol style='margin:8px 0 0 18px'>"
                 + "".join(f"<li>{s}</li>" for s in steps) + "</ol></div>"))

try:
    GOOGLE_API_KEY = (userdata.get("GOOGLE_API_KEY") or "").strip()   # your class pass (CODE:Your Name), or your own Gemini key
except Exception as e:
    help_box("Colab can't see your class pass yet", [
        "Click the 🔑 key icon in the left sidebar.",
        "Add a secret named exactly <code>GOOGLE_API_KEY</code>, value = your class pass, like <code>CODE:Maya R</code>.",
        "Switch on <b>Notebook access</b> for it.",
        "Run this cell again."])
    raise Stop(f"No class pass found ({type(e).__name__}). Follow the steps above.") from None
if ":" not in GOOGLE_API_KEY and not GOOGLE_API_KEY.startswith("AIza"):
    help_box("That doesn't look like a class pass", [
        "A class pass has a colon in the middle: the class code, a colon, your first name and last initial. Like <code>CODE:Maya R</code>.",
        "Fix the value in 🔑 Secrets, then run this cell again."])
    raise Stop("The class pass needs a colon. Fix it in Secrets and run again.")

from google import genai
from google.genai import types
from fastmcp import FastMCP, Client

# A class pass goes through the Sawyer AI Sandbox: no personal key, no credit card. A personal key goes straight to Google.
SANDBOX = {"base_url": "https://sawyer-ai-sandbox.vercel.app/api/gemini"} if ":" in GOOGLE_API_KEY else None
client = genai.Client(api_key=GOOGLE_API_KEY, http_options=SANDBOX)
MODEL  = "gemini-2.5-flash"      # a brand-new key may be told 2.5 is "not found" → use "gemini-3.8-flash" (also free)
print("✅ Gemini ready · ✅ MCP ready")

## 🔨 Build the server

The same Open-Meteo code from Session 5: free, no key, any city on earth. **Find the one new line.**

In [ ]:
from fastmcp import FastMCP
import urllib.request, urllib.parse, json, datetime

mcp = FastMCP("Weather")

def _get(url, **params):
    with urllib.request.urlopen(url + "?" + urllib.parse.urlencode(params), timeout=10) as r:
        return json.load(r)

@mcp.tool                     # ← the only new line. It turns a function into an MCP tool.
def get_weather(city: str, date: str = "") -> str:
    """Weather forecast for any city in the world.
    date: YYYY-MM-DD within the next 14 days. Leave empty for today and tomorrow."""
    places = _get("https://geocoding-api.open-meteo.com/v1/search", name=city, count=1).get("results")
    if not places:
        return f"No place called '{city}' found."
    p = places[0]
    w = _get("https://api.open-meteo.com/v1/forecast", latitude=p["latitude"], longitude=p["longitude"],
             daily="temperature_2m_max,temperature_2m_min,precipitation_probability_max",
             temperature_unit="fahrenheit", timezone="auto", forecast_days=16)["daily"]
    days = range(2) if not date else [i for i, d in enumerate(w["time"]) if d == date]
    if not days:
        return f"No forecast for {date} — I can only see 16 days ahead."
    lines = [f"{w['time'][i]}: high {w['temperature_2m_max'][i]:.0f}°F, low {w['temperature_2m_min'][i]:.0f}°F, "
             f"{w['precipitation_probability_max'][i]}% chance of rain" for i in days]
    return f"{p['name']}, {p.get('country', '?')}\n" + "\n".join(lines)

print("✅ MCP server built:", mcp.name)

## 📡 Turn on the X-ray

MCP is just messages, little pieces of JSON going back and forth. This cell draws each one as a chat bubble the moment it crosses the wire: **blue** is the agent talking, **purple** is the server answering. The grey line under a bubble is the actual JSON that was sent.

In [ ]:
# 📡 THE X-RAY: draws every MCP message as it crosses the wire. Plumbing, so you don't need to read this cell.
import html, json
from IPython.display import display, HTML
from fastmcp.server.middleware import Middleware

COLORS = {"agent": ("#ecfeff", "#0e7490"), "server": ("#f5f3ff", "#6d28d9"), "you": ("#f3f4f6", "#374151"),
          "answer": ("#f0fdf4", "#15803d"), "blocked": ("#fef2f2", "#b91c1c")}

def bubble(kind, who, text, wire=None):
    bg, fg = COLORS[kind]
    side = "flex-end" if kind in ("server", "blocked") else "flex-start"
    wire_html = (f'<div style="font-family:ui-monospace,monospace;font-size:12px;color:#475569;margin-top:8px;'
                 f'white-space:pre-wrap">{html.escape(wire)}</div>') if wire else ""
    display(HTML(f'''<div style="display:flex;justify-content:{side};margin:8px 0">
      <div style="max-width:78%;background:{bg};color:#111827;border:1.5px solid {fg};border-radius:16px;padding:10px 16px;
                  font-family:system-ui,sans-serif;font-size:15px;line-height:1.45">
        <div style="color:{fg};font-weight:700;font-size:11px;letter-spacing:.1em;text-transform:uppercase;margin-bottom:4px">{who}</div>
        {html.escape(text).replace(chr(10), "<br>")}{wire_html}</div></div>'''))

def note(text):
    display(HTML(f'<div style="text-align:center;color:#64748b;font:italic 12px system-ui,sans-serif;margin:2px 0">{html.escape(text)}</div>'))

class XRay(Middleware):
    last = None
    async def on_request(self, context, call_next):
        before, self.last = self.last, context.method
        if context.method == "tools/list" and before == "tools/call":
            note("(the client quietly re-reads the menu to check the result's format: real traffic, nothing new)")
            return await call_next(context)
        if context.method == "tools/list":
            bubble("agent", "🤖 agent → MCP server", '① "What can you do?"', '{"jsonrpc": "2.0", "method": "tools/list"}')
            tools = await call_next(context)
            bubble("server", "🔌 MCP server → agent", "Here's my menu:\n" +
                   "\n".join(f"• {t.name}: {(t.description or '').splitlines()[0]}" for t in tools))
            return tools
        if context.method == "tools/call":
            name, args = context.message.name, context.message.arguments or {}
            wire = {"jsonrpc": "2.0", "method": "tools/call", "params": {"name": name, "arguments": args}}
            bubble("agent", "🤖 agent → MCP server", f'② "Do this: {name}"', json.dumps(wire, ensure_ascii=False))
            try:
                result = await call_next(context)
            except Exception as e:
                bubble("blocked", "🔌 MCP server → agent", f"③ ⛔ {e}")
                raise
            bubble("server", "🔌 MCP server → agent", "③ " + result.content[0].text)
            return result
        return await call_next(context)

mcp.add_middleware(XRay())
print("📡 X-ray on: every message between agent and server will be drawn below the cell that sends it")

## ① *"What can you do?"*

In Session 4 you wrote about 15 lines of JSON to describe your tool to Gemini. **Nobody wrote this one.** MCP built the menu from the function's name, its docstring, and its type hints.

In [ ]:
async with Client(mcp) as server:
    menu = await server.list_tools()

print("What the agent actually receives (nobody typed this):\n")
print(json.dumps({"name": menu[0].name, "description": menu[0].description, "inputSchema": menu[0].input_schema}, indent=2))

## ② *"Do this."* No AI yet

Call the tool through the server, exactly the way an agent would.

In [ ]:
async with Client(mcp) as server:
    await server.call_tool("get_weather", {"city": "Boston"})

## 🤖 Plug Gemini in

This is **the Session 4 loop** with one change: the menu comes *from the server*, and every call goes *through the server*. Gemini never touches Open-Meteo. It only speaks MCP. Watch the bubbles.

In [ ]:
async def ask(question):
    bubble("you", "🙋 you → Gemini", question)
    async with Client(mcp) as server:
        menu = await server.list_tools()                                     # ① "What can you do?"
        tools = [types.Tool(function_declarations=[
            types.FunctionDeclaration(name=t.name, description=t.description,
                                      parameters_json_schema={k: v for k, v in t.input_schema.items() if k != "additionalProperties"})
            for t in menu])]
        chat = client.aio.chats.create(model=MODEL, config=types.GenerateContentConfig(tools=tools))
        reply = await chat.send_message(question)
        while reply.function_calls:                                          # the Session 4 loop
            answers = []
            for call in reply.function_calls:
                try:
                    result = (await server.call_tool(call.name, dict(call.args))).data   # ② "Do this." ③ the result
                except Exception as e:
                    result = f"ERROR: {e}"                                   # honest errors, just like Session 4
                answers.append(types.Part.from_function_response(name=call.name, response={"result": result}))
            reply = await chat.send_message(answers)
    bubble("answer", "🤖 Gemini → you", reply.text)

In [ ]:
await ask("Do I need an umbrella in Boston tomorrow?")

## ➕ Add a tool in four lines

No JSON, no registration, no change to `ask()`. The next time an agent asks *"What can you do?"*, the menu is longer.

In [ ]:
import datetime

@mcp.tool
def days_until(date: str) -> str:
    """How many days from today until a date. date: YYYY-MM-DD."""
    return f"{(datetime.date.fromisoformat(date) - datetime.date.today()).days} days from today ({datetime.date.today()})"

In [ ]:
await ask("Our midterm is due November 2, 2026. How many days away is that, and should I expect rain in Boston that day?")

## 🇸🇪 The twist

Remember `"Bostn"` from Session 5?

In [ ]:
async with Client(mcp) as server:
    await server.call_tool("get_weather", {"city": "Bostn"})

**Båstnäs, Sweden.** The protocol delivered the wrong answer perfectly. MCP standardizes the *plug*. It doesn't make the tool any smarter.

---

## 🔒 Bonus: a protocol is not a permission (try it tonight)

In class the Pizza server asked *"who approved this?"* Here's the same move in code. The gate lives **inside the server**, so no agent that plugs in can skip it, however politely it asks. It's the same trust layer as your midterm.

In [ ]:
from fastmcp.exceptions import ToolError

@mcp.tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email on the user's behalf."""
    return f"Email sent to {to}."            # simulated, like your midterm

class Gate(Middleware):                      # the gate lives in the SERVER, so no agent can skip it
    async def on_call_tool(self, context, call_next):
        if context.message.name == "send_email":
            raise ToolError("BLOCKED: nobody approved send_email. Plugging in is not permission.")
        return await call_next(context)

mcp.add_middleware(Gate())
print("🔒 send_email is on the menu, behind a gate")

In [ ]:
await ask("Email my landlord at landlord@example.com that the heat is broken again.")

---

## 🎓 Your turn: build your first MCP server (5 minutes, tonight)

Your midterm agent already has tools. Each one is a regular Python function. Make one of them an MCP server tool:

1. Paste **one tool from your midterm** over `my_tool` below. Keep its docstring, because **the docstring is the menu**.
2. Leave `@mcp.tool` on top. That's the whole build.
3. Run the cell. The 📡 X-ray shows your tool on the menu next to `get_weather`.
4. Then ask Gemini something only your tool can answer: `await ask("...")`.

If Gemini picks your tool without being told its name, your docstring did its job. If it doesn't, rewrite the docstring, not the question.

In [ ]:
@mcp.tool
def my_tool(text: str) -> str:
    """Replace me with one tool from your midterm. Say what it does and when to use it."""
    return f"(placeholder) you asked about: {text}"

async with Client(mcp) as server:      # 📡 watch your tool join the menu
    await server.list_tools()

# await ask("a question only YOUR tool can answer")

### What you just saw
1. **A server** is your function plus `@mcp.tool`. MCP writes the menu for you.
2. **MCP is just messages**: `tools/list` and `tools/call`, in JSON. You saw every one.
3. **Any agent** that speaks MCP can plug in. This same server works in Claude Desktop, Cursor, VS Code, or Gemini CLI with no changes.
4. **Underneath, there's still an API** (Open-Meteo). MCP doesn't replace APIs. It wraps them so any AI can use them.
5. **The plug is not the permission.** The gate decides, not the protocol.

When a vendor says *"it's MCP-compatible,"* ask: **compatible with what, and who said yes?**